# nb09 — Full Ablation: Loss × Augmentation × Featurizer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TigreGotico/ww-trainer/blob/dev/notebooks/nb09_ablation.ipynb)

> **Which featurizer, which loss, and how much augmentation give the best wake-word model?**  \
> This notebook answers that empirically by training a model for *every combination* of three\
> design choices and comparing them in heatmaps and bar charts.

---

## What is an "ablation"?

An **ablation study** systematically changes one ingredient of a system at a time (or, here,
sweeps a full grid of combinations) and measures the effect on performance. The name comes
from "ablating" (removing/altering) a component to see how much it mattered. The goal is not
a single best model but *understanding*: which choices move the needle, and which do not.

This notebook ablates three independent design axes:

| Axis | What it controls | Levels swept |
|------|------------------|--------------|
| **Featurizer** | How audio becomes features | `mfcc`, `filterbank`, `sincnet` |
| **Loss** | What the model is optimised to do | `bce`, `focal`, `rppl`, `arcface` |
| **Augmentation** | How much synthetic noise is mixed in | `none`, `bg_noise`, `full` |

Every combination is one experiment, so the default grid is 3 × 4 × 3 = **36 runs**.

### The jargon, defined

- **Featurizer** — the front-end converting raw audio into numeric features.
  - `mfcc` — classic hand-designed cepstral features (cheap, no learnable parameters).
  - `filterbank` — log-mel filterbank energies (also hand-designed, slightly richer).
  - `sincnet` — a *learnable* filterbank whose band-pass filters are trained from data.
- **Loss function** — the objective the optimiser minimises during training.
  - `bce` — Binary Cross-Entropy, the standard "is it the wake word, yes/no?" loss.
  - `focal` — Focal loss; down-weights easy examples so training focuses on the hard,
    confusable ones. Useful when negatives vastly outnumber positives.
  - `rppl` — Robust Prototype & Diversity Loss; a composite loss that, in addition to BCE,
    pulls wake-word embeddings toward a stable "prototype" vector and spreads confusable
    negatives apart.
  - `arcface` — Additive Angular Margin loss; enforces an angular gap between the wake and
    non-wake classes in embedding space (borrowed from face/speaker verification).
- **Augmentation** — mixing synthetic distortions into training audio so the model
  generalises to real, messy rooms.
  - `none` — clean audio only.
  - `bg_noise` — background-noise mixing only.
  - `full` — background noise **plus** music and room reverberation (impulse responses).

---

## How to read the results

This notebook produces three views, in increasing detail. Each is explained again above the
cell that creates it.

1. **Heatmap (Cell 7)** — F1 as a grid of *featurizer × loss*, averaged over augmentation
   levels. Brighter = higher F1. Compare **rows** to judge featurizer impact and **columns**
   to judge loss impact.
2. **Augmentation impact chart (Cell 8)** — for each featurizer, the F1 lift going from
   `none → bg_noise → full`. This tells you whether collecting augmentation data is worth
   the effort.
3. **Embedding quality (Cell 9)** — for the top-5 configurations, two geometry metrics:
   - **Fisher ratio** = between-class variance ÷ within-class variance. Higher means the
     wake and non-wake clusters are better separated.
   - **Silhouette score** (range −1 … 1) = how cohesive each cluster is versus how far apart
     the clusters are. Higher is better.

> **What conclusion to draw:** a good design choice raises F1 *and* the embedding-quality
> metrics together. If a config has high F1 but poor Fisher/silhouette, it may be overfitting
> the test split rather than learning a clean decision boundary — treat it with suspicion.

---

## Runtime budget

Each run takes roughly 8–15 min on a Kaggle T4 GPU, so the full 36-cell grid is **~5–9 h**.
The grid is **resume-safe**: with `SKIP_COMPLETED=true` every finished run is cached to disk,
so you can stop, restart, and continue across multiple sessions without repeating work.

| Platform | Per run | Full grid (36) |
|----------|---------|----------------|
| Kaggle T4 GPU | ~8–15 min | ~5–9 h (multi-session) |
| Local CPU | ~30–90 min | ~20–50 h (multi-session) |

**To shrink the experiment:** lower `EPOCHS` (e.g. `10`) for a faster but noisier sweep, or
trim the axis lists (e.g. `LOSSES=bce,focal`) to cut the number of runs.

## Cell 1 — Configuration

**The cell to edit.** Set the wake phrase and choose which levels of each axis to sweep.
The three axis lists (`FEATURIZERS`, `LOSSES`, `AUGMENT_LEVELS`) are comma-separated; their
product is the number of runs, printed as `N_CELLS`.

Every value can also come from an environment variable of the same name, so you can launch
different sweeps without editing code.

- `FEATURIZERS` — any of `mfcc`, `filterbank`, `sincnet`, `gammatone`, `micro`, `delta`.
- `LOSSES` — any loss name `ww_trainer` understands (e.g. `bce`, `focal`, `rppl`, `arcface`).
- `AUGMENT_LEVELS` — `none`, `bg_noise`, `full`.
- `EPOCHS` — per run. The default `20` is a sensible accuracy/speed trade-off; drop to `10`
  for a quick noisy pass.
- `SKIP_COMPLETED` — keep `true` to make the grid resumable across sessions.

> **Tip:** start small (e.g. `FEATURIZERS=mfcc`, `LOSSES=bce,focal`) to validate the whole
> pipeline end-to-end in minutes, then widen the axes for the full study.

In [ ]:
import os

# ── Core ──────────────────────────────────────────────────────────────────────
WAKE_WORD         = os.environ.get("WAKE_WORD",         "hey jarvis")
OUTPUT_DIR        = os.environ.get("OUTPUT_DIR",        "./ww_output")
DEVICE            = os.environ.get("DEVICE",            "auto")
SEED              = int(os.environ.get("SEED",          "42"))

# ── Ablation axes ─────────────────────────────────────────────────────────────
FEATURIZERS       = os.environ.get("FEATURIZERS",       "mfcc,filterbank,sincnet").split(",")
LOSSES            = os.environ.get("LOSSES",            "bce,focal,rppl,arcface").split(",")
AUGMENT_LEVELS    = os.environ.get("AUGMENT_LEVELS",    "none,bg_noise,full").split(",")

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS            = int(os.environ.get("EPOCHS",        "20"))
BATCH_SIZE        = int(os.environ.get("BATCH_SIZE",    "16"))
SKIP_COMPLETED    = os.environ.get("SKIP_COMPLETED",    "true").lower() == "true"

# ── Dataset ───────────────────────────────────────────────────────────────────
N_POSITIVE        = int(os.environ.get("N_POSITIVE",    "400"))
LANG              = os.environ.get("LANG",              "en")
ADVERSARIAL       = os.environ.get("ADVERSARIAL",       "true").lower() == "true"
DOWNLOAD_AUGMENT  = os.environ.get("DOWNLOAD_AUGMENT",  "true").lower() == "true"
REUSE_DATASET     = os.environ.get("REUSE_DATASET",     "true").lower() == "true"
CUSTOM_TRAIN_CSV  = os.environ.get("CUSTOM_TRAIN_CSV",  "")
CUSTOM_TEST_CSV   = os.environ.get("CUSTOM_TEST_CSV",   "")

# ── MLflow (optional) ────────────────────────────────────────────────────────
MLFLOW_URI        = os.environ.get("MLFLOW_URI",        "")
MLFLOW_SECRET     = os.environ.get("MLFLOW_SECRET",     "MLFLOW_TOKEN")
MLFLOW_EXPERIMENT = os.environ.get("MLFLOW_EXPERIMENT", "ww_ablation")

N_CELLS = len(FEATURIZERS) * len(LOSSES) * len(AUGMENT_LEVELS)
print(f"Ablation grid: {len(FEATURIZERS)} featurizers × {len(LOSSES)} losses × {len(AUGMENT_LEVELS)} augment = {N_CELLS} cells")
print(f"Featurizers : {FEATURIZERS}")
print(f"Losses      : {LOSSES}")
print(f"Augmentation: {AUGMENT_LEVELS}")
print(f"Epochs/cell : {EPOCHS}")

## Cell 2 — Install dependencies and detect platform

Installs `ww_trainer`, the audio/ML stack, `seaborn` (for the heatmap), and the OVOS plugins
used during dataset generation. Already-installed packages are skipped, so re-running is
safe.

It also detects the platform (Kaggle / Paperspace / Colab / local) and caps CPU threads for
stable, comparable timing across runs — important in an ablation where you compare numbers.

In [ ]:
import subprocess, sys, os

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("torch", "torchaudio", "soundfile", "numpy", "scikit-learn",
     "matplotlib", "pandas", "seaborn", "librosa", "onnx", "onnxruntime",
     "click", "tqdm")
_pip("ovos-plugin-manager", "ovos-tts-plugin-edge-tts",
     "git+https://github.com/TigreGotico/vadonnx.git", "datasets")
try:
    import ww_trainer
except ImportError:
    _pip("ww_trainer")

_platform = (
    "kaggle"     if os.path.exists("/kaggle")     else
    "paperspace" if os.path.exists("/notebooks")  else
    "colab"      if "google.colab" in sys.modules else
    "local"
)

import torch
torch.set_num_threads(min(12, os.cpu_count() or 4))
os.environ.setdefault("OMP_NUM_THREADS", str(min(12, os.cpu_count() or 4)))

print(f"Platform : {_platform} | CUDA: {torch.cuda.is_available()} | threads: {torch.get_num_threads()}")

## Cell 3 — MLflow setup (optional)

Configures **MLflow** experiment tracking, if you use it. MLflow is a tool that logs each
run's parameters and metrics to a server so you can browse and compare them in a web UI —
handy when you have dozens of ablation runs.

This is entirely optional. If `MLFLOW_URI` is empty (the default), tracking is disabled and
the ablation runs exactly the same way; results are still saved to local JSON files. On
Kaggle, the MLflow token is read from a Secret named by `MLFLOW_SECRET`.

In [ ]:
import os

# ── MLflow setup ──────────────────────────────────────────────────────────────
if _platform == "kaggle" and MLFLOW_SECRET:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(MLFLOW_SECRET)
        os.environ["MLFLOW_TRACKING_TOKEN"] = token
        print(f"MLflow token injected from Kaggle Secret '{MLFLOW_SECRET}'")
    except Exception as e:
        print(f"MLflow secret not found: {e}")

if MLFLOW_URI:
    os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_URI
    try:
        import mlflow
        mlflow.set_tracking_uri(MLFLOW_URI)
        mlflow.set_experiment(MLFLOW_EXPERIMENT)
        print(f"MLflow experiment: {MLFLOW_EXPERIMENT!r} at {MLFLOW_URI}")
    except Exception as e:
        print(f"MLflow warning: {e}")
else:
    print("MLFLOW_URI not set — tracking disabled")

## Cell 4 — Prepare the dataset (built once, shared by every run)

The ablation trains many models, but they must all see the **same data** — otherwise the
comparison is meaningless. This cell builds (or reuses) one dataset and one train/test split
that every run in the grid reuses.

As in the other notebooks, data comes from one of three sources in priority order:

1. **`CUSTOM_TRAIN_CSV`** — your own labelled CSV (auto-split 80/20 if no test CSV given).
2. **`REUSE_DATASET=true`** with an existing dataset → reused instantly.
3. Otherwise → generate fresh (TTS positives + downloaded negatives + augmentation audio).

The augmentation folders (background noise, music, room reverb) are stored in
`_aug_kwargs_full`; Cell 6 selects from them according to each run's augmentation level. The
disk-space check expects ~5 GB free, since 36 runs each save a small model
(~200 MB total across the grid).

In [ ]:
import shutil
from pathlib import Path

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
# Ablation needs ~200 MB per cell → 36 cells × 200 MB = ~7 GB
assert free_gb > 5, f"Only {free_gb:.1f} GB free — need at least 5 GB for full ablation."
print(f"Disk free: {free_gb:.1f} GB  (36 cells × ~200 MB ≈ 7 GB needed for full run)")

_aug_kwargs_full = {}

if CUSTOM_TRAIN_CSV:
    import random, csv as _csv
    from ww_trainer.utils import read_dataset_csv
    train_csv = Path(CUSTOM_TRAIN_CSV)
    test_csv  = Path(CUSTOM_TEST_CSV) if CUSTOM_TEST_CSV else None
    if test_csv is None:
        split_dir = Path(OUTPUT_DIR) / "dataset_split"
        split_dir.mkdir(parents=True, exist_ok=True)
        split_train = split_dir / "train.csv"
        split_test  = split_dir / "test.csv"
        if not split_train.exists():
            rows = read_dataset_csv(train_csv)
            random.seed(SEED); random.shuffle(rows)
            cut = int(len(rows) * 0.8)
            for p, rs in [(split_train, rows[:cut]), (split_test, rows[cut:])]:
                with open(p, "w", newline="") as f:
                    _csv.writer(f).writerows(rs)
        train_csv, test_csv = split_train, split_test
else:
    from ww_trainer.datagen import DatagenConfig, run_datagen_pipeline, DatagenResult, normalize_wake_word
    dataset_dir = Path(OUTPUT_DIR) / "dataset"
    _train_csv_check = dataset_dir / "train" / "metadata.csv"
    if REUSE_DATASET and _train_csv_check.exists():
        slug = normalize_wake_word(WAKE_WORD)
        _dr = DatagenResult(
            train_csv=dataset_dir / "train" / "metadata.csv",
            test_csv=dataset_dir / "test" / "metadata.csv",
            positives_dir=dataset_dir / slug / "positives",
            negatives_dir=dataset_dir / slug / "negatives",
            bg_noise_dir=dataset_dir / "augmentation" / "bg_noise",
            music_dir=dataset_dir / "augmentation" / "music",
            rir_dir=dataset_dir / "augmentation" / "rir",
        )
        print(f"Reusing dataset at {dataset_dir}")
    else:
        _dr = run_datagen_pipeline(DatagenConfig(
            wake_word=WAKE_WORD, output_dir=dataset_dir,
            n_positive=N_POSITIVE, lang=LANG,
            adversarial=ADVERSARIAL, vad_trim=True,
            download_augmentation=DOWNLOAD_AUGMENT, seed=SEED,
        ))
    train_csv, test_csv = _dr.train_csv, _dr.test_csv
    for attr, key in [("bg_noise_dir", "bg_noise_folder"),
                      ("music_dir", "music_folder"),
                      ("rir_dir", "rir_folder")]:
        d = getattr(_dr, attr, None)
        if d and Path(d).exists():
            _aug_kwargs_full[key] = str(d)

print(f"train_csv: {train_csv}")
print(f"test_csv : {test_csv}")

## Cell 5 — Run the ablation grid (the long-running cell)

This is the heart of the notebook: a triple loop over featurizer × loss × augmentation that
trains one model per combination. **Expect this cell to run for hours** on the full grid.

For each combination it:

1. Maps the featurizer name to a model *tier* via `FEAT_TO_TIER` (e.g. `mfcc → small`,
   `filterbank → filterbank_small`). A *tier* bundles a featurizer with a matching
   classifier head and size budget.
2. Selects the augmentation folders for the current level (`none` = none, `bg_noise` = noise
   only, `full` = all).
3. Calls `train_from_wakeword(...)` with `losses_cfg=[{"name": loss, "weight": 1.0}]` so the
   chosen loss drives training, reusing the shared dataset.
4. Saves a per-run JSON (`<feat>__<loss>__<augment>.json`) recording F1, precision, recall,
   runtime, and the exported ONNX paths.

**Resume safety.** With `SKIP_COMPLETED=true`, any combination whose JSON already exists is
skipped and its cached result loaded — this is what lets you spread the grid over several
sessions. Errors in one run are caught, recorded with `status: "error"`, and do not abort
the rest of the grid.

> You can re-run this cell as many times as you like; it only trains the combinations that
> have not completed yet.

In [ ]:
import json, time
from pathlib import Path
from ww_trainer.quickstart import train_from_wakeword

# ── Ablation grid loop (resumable) ────────────────────────────────────────────
# Featurizer maps to tier: mfcc→small, filterbank→filterbank_small, sincnet→sincnet_small

FEAT_TO_TIER = {
    "mfcc":       "small",
    "filterbank": "filterbank_small",
    "sincnet":    "sincnet_small",
    "gammatone":  "gammatone_small",
    "micro":      "micro",
    "delta":      "delta_micro",
}

def _aug_kwargs_for_level(level):
    if level == "none":
        return {}
    if level == "bg_noise":
        return {k: v for k, v in _aug_kwargs_full.items() if "bg_noise" in k}
    return dict(_aug_kwargs_full)

results_dir = Path(OUTPUT_DIR) / "ablation_results"
results_dir.mkdir(parents=True, exist_ok=True)
all_results = []

total = N_CELLS
cell_idx = 0

for feat in FEATURIZERS:
    tier = FEAT_TO_TIER.get(feat, "small")
    for loss in LOSSES:
        for augment in AUGMENT_LEVELS:
            cell_idx += 1
            cell_key  = f"{feat}__{loss}__{augment}"
            result_file = results_dir / f"{cell_key}.json"
            model_subdir = Path(OUTPUT_DIR) / "ablation_models" / cell_key

            print(f"\n[{cell_idx}/{total}] feat={feat!r}  loss={loss!r}  aug={augment!r}")

            if SKIP_COMPLETED and result_file.exists():
                saved = json.loads(result_file.read_text())
                print(f"  SKIP: F1={saved.get('f1', 0):.4f}")
                all_results.append(saved)
                continue

            aug_kw = _aug_kwargs_for_level(augment)
            t0 = time.time()
            try:
                r = train_from_wakeword(
                    WAKE_WORD, str(model_subdir),
                    tier=tier,
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    device=DEVICE,
                    seed=SEED,
                    reuse_dataset=True,
                    losses_cfg=[{"name": loss, "weight": 1.0}],
                    **aug_kw,
                )
                elapsed = time.time() - t0
                row = {
                    "featurizer": feat,
                    "tier": tier,
                    "loss": loss,
                    "augment": augment,
                    "f1": r.metrics.get("f1", 0.0),
                    "precision": r.metrics.get("precision", 0.0),
                    "recall": r.metrics.get("recall", 0.0),
                    "elapsed_s": round(elapsed, 1),
                    "feat_onnx": str(model_subdir / "model" / "best_f1_featurizer.onnx"),
                    "head_onnx": str(r.best_onnx_path) if r.best_onnx_path else "",
                    "status": "ok",
                }
                print(f"  DONE: F1={row['f1']:.4f}  ({elapsed:.0f}s)")
            except Exception as exc:
                elapsed = time.time() - t0
                row = {
                    "featurizer": feat, "tier": tier,
                    "loss": loss, "augment": augment,
                    "f1": 0.0, "precision": 0.0, "recall": 0.0,
                    "elapsed_s": round(elapsed, 1),
                    "feat_onnx": "", "head_onnx": "",
                    "status": f"error: {exc}",
                }
                print(f"  ERROR: {exc}")

            result_file.write_text(json.dumps(row, indent=2))
            all_results.append(row)

print(f"\nGrid complete: {sum(1 for r in all_results if r['status']=='ok')}/{len(all_results)} succeeded.")

## Cell 6 — Heatmap: featurizer × loss

Reloads **all** per-run JSON files from disk (so it works on a partial grid too) and builds
the headline view: a heatmap of mean F1 for each *featurizer × loss* pair, averaged over the
three augmentation levels.

**How to read it:**

- Each cell is the average F1 for that featurizer/loss combination. Brighter (warmer) =
  higher F1.
- **Scan across a row** to see how much the *loss* changes results for a fixed featurizer.
- **Scan down a column** to see how much the *featurizer* changes results for a fixed loss.
- A row or column that is uniformly bright means that choice is robust; a patchy one means
  the choice interacts strongly with the other axis.

The cell also prints the underlying pivot table and the **top-5 configurations** by F1 —
your shortlist of candidates. Reading the table alongside the heatmap is the quickest way to
spot the winning region of the grid.

> If `seaborn` is unavailable the heatmap is skipped, but the numeric pivot table still
> prints — no information is lost.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ── Reload all results from disk ──────────────────────────────────────────────
# (handles partial runs and multi-session resumption)

_loaded = []
for f in sorted((Path(OUTPUT_DIR) / "ablation_results").glob("*.json")):
    _loaded.append(json.loads(f.read_text()))

df = pd.DataFrame(_loaded)
df_ok = df[df["status"] == "ok"].copy()

print(f"Loaded {len(df)} cells ({len(df_ok)} successful)")

if df_ok.empty:
    print("No successful cells yet — run Cell 6 first.")
else:
    # ── Pivot table: featurizer × loss (mean F1 over augment levels) ──────
    pivot = df_ok.pivot_table(
        index="featurizer", columns="loss", values="f1", aggfunc="mean"
    )
    print("\nF1 pivot (mean over augmentation levels):")
    print(pivot.round(4).to_string())

    # ── Heatmap ───────────────────────────────────────────────────────────
    try:
        import seaborn as sns
        fig, ax = plt.subplots(figsize=(max(6, len(LOSSES) * 1.5),
                                        max(3, len(FEATURIZERS) * 1.2)))
        sns.heatmap(
            pivot, annot=True, fmt=".3f", cmap="YlOrRd",
            vmin=0, vmax=1, ax=ax,
            linewidths=0.5, linecolor="white",
        )
        ax.set_title(f"Ablation heatmap: featurizer × loss (mean F1)\n{WAKE_WORD!r}")
        ax.set_xlabel("Loss function")
        ax.set_ylabel("Featurizer")
        plt.tight_layout()
        heatmap_path = str(Path(OUTPUT_DIR) / "ablation_heatmap.png")
        plt.savefig(heatmap_path, dpi=120, bbox_inches="tight")
        plt.show()
        print(f"Heatmap saved: {heatmap_path}")
    except ImportError:
        print("seaborn not installed — skipping heatmap. Run: pip install seaborn")
        print(pivot.round(4).to_string())

    # Top 5
    top5 = df_ok.nlargest(5, "f1")[["featurizer","loss","augment","f1","precision","recall"]]
    print("\nTop 5 configurations:")
    print(top5.to_string(index=False))

## Cell 7 — Augmentation impact

This chart isolates the effect of *augmentation* alone. For each featurizer it plots mean F1
at each augmentation level (`none`, `bg_noise`, `full`), averaged over all losses, as three
grouped bars.

**How to read it and what to conclude:**

- Rising bars from `none → bg_noise → full` mean augmentation helps — the model becomes more
  robust as you add noise, music, and reverb. This is the common and desirable pattern.
- Flat bars mean augmentation barely matters for that featurizer on this data — collecting
  augmentation audio may not be worth the effort.
- Falling bars (rare) suggest the augmentation is too aggressive for the task and is hurting
  more than helping.

The printed `f1_lift_none_to_full` column quantifies the gain directly: a positive number is
the F1 improvement augmentation bought you. Use it to decide whether the data-collection
cost of augmentation pays off for *your* wake word.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# ── Augmentation impact bar chart ─────────────────────────────────────────────
# For each featurizer, show mean F1 at each augmentation level (averaged over losses).
# This isolates the effect of augmentation from the effect of loss/featurizer.

if df_ok.empty:
    print("No data yet.")
else:
    aug_pivot = df_ok.pivot_table(
        index="featurizer", columns="augment", values="f1", aggfunc="mean"
    )

    # Ensure augment columns are in the right order
    _aug_order = [a for a in ["none", "bg_noise", "full"] if a in aug_pivot.columns]
    aug_pivot = aug_pivot[_aug_order]

    print("Augmentation impact (mean F1 over losses):")
    print(aug_pivot.round(4).to_string())

    n_feats = len(aug_pivot)
    n_augs  = len(_aug_order)
    x = np.arange(n_feats)
    width = 0.25
    colors = ["#d9d9d9", "#6baed6", "#2171b5"]

    fig, ax = plt.subplots(figsize=(max(7, n_feats * 2), 4))
    for i, (aug_level, color) in enumerate(zip(_aug_order, colors)):
        if aug_level in aug_pivot.columns:
            vals = aug_pivot[aug_level].values
            bars = ax.bar(x + (i - n_augs/2 + 0.5) * width, vals,
                          width, label=aug_level, color=color, edgecolor="white")

    ax.set_xticks(x)
    ax.set_xticklabels(aug_pivot.index, rotation=15)
    ax.set_ylabel("F1 (mean over losses)")
    ax.set_title(f"Augmentation impact by featurizer — {WAKE_WORD!r}")
    ax.set_ylim(0, 1.05)
    ax.legend(title="Augmentation", fontsize=9)

    plt.tight_layout()
    aug_path = str(Path(OUTPUT_DIR) / "augmentation_impact.png")
    plt.savefig(aug_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Augmentation impact chart saved: {aug_path}")

    # Delta: none → full
    if "none" in aug_pivot.columns and "full" in aug_pivot.columns:
        delta = (aug_pivot["full"] - aug_pivot["none"]).rename("f1_lift_none_to_full")
        print("\nF1 lift (none → full augmentation):")
        print(delta.round(4).to_string())

## Cell 8 — Embedding quality of the top-5 configs

F1 measures *decisions*; this cell measures the *geometry* underneath them. For the five
highest-F1 configurations it extracts featurizer embeddings on the test set and computes two
cluster-separation metrics:

- **Fisher ratio** = between-class variance ÷ within-class variance. It rewards wake and
  non-wake clusters whose means are far apart (large numerator) and whose points are tightly
  packed (small denominator). **Higher is better.**
- **Silhouette score** (range −1 … 1, from scikit-learn) = for each point, how close it sits
  to its own cluster versus the other cluster. Near `1` = clean, well-separated clusters;
  near `0` = overlapping; negative = points sitting in the wrong cluster. **Higher is
  better.**

**What conclusion to draw:** the most trustworthy configurations score well on F1 *and* on
both geometry metrics — that combination indicates the model learned a genuinely separable
representation rather than memorising the test split. If a top-F1 config has weak Fisher /
silhouette, prefer a slightly lower-F1 config with cleaner geometry; it usually generalises
better to real-world audio.

The cell prints a summary table and a three-panel bar chart (F1, Fisher, silhouette) so you
can eyeball whether the rankings agree across all three metrics.

In [ ]:
import numpy as np
import csv
import torchaudio
import onnxruntime as ort
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import silhouette_score

# ── Embedding quality: Fisher ratio + silhouette for top-5 configs ─────────────
# Fisher ratio = inter-class variance / intra-class variance.
# Silhouette score measures cluster cohesion vs separation in [-1, 1].
# Both metrics are computed on featurizer embeddings (not classifier outputs).

if df_ok.empty:
    print("No successful runs yet.")
else:
    # Load test samples once
    _wavs_eq, _labels_eq = [], []
    with open(test_csv) as f:
        for row in csv.reader(f):
            if len(row) < 2 or not Path(row[0]).exists():
                continue
            wav, sr = torchaudio.load(row[0])
            if sr != 16000:
                wav = torchaudio.functional.resample(wav, sr, 16000)
            _wavs_eq.append(wav.mean(0).numpy().astype(np.float32))
            _labels_eq.append(int(row[1].strip()))
            if len(_wavs_eq) >= 200:
                break
    _labels_arr = np.array(_labels_eq)
    print(f"Test samples: {len(_wavs_eq)} ({(_labels_arr==1).sum()} pos, {(_labels_arr==0).sum()} neg)")

    def _extract_embs(feat_onnx_path, wavs):
        sess = ort.InferenceSession(str(feat_onnx_path), providers=["CPUExecutionProvider"])
        in_name = sess.get_inputs()[0].name
        embs = []
        for wav in wavs:
            out = sess.run(None, {in_name: wav[np.newaxis, :]})[0]
            embs.append(out.mean(axis=1).ravel() if out.ndim == 3 else out.ravel())
        return np.array(embs)

    def _fisher_ratio(embs, labels):
        """Fisher ratio: inter-class / intra-class variance (mean over dims)."""
        pos = embs[labels == 1]
        neg = embs[labels == 0]
        if len(pos) < 2 or len(neg) < 2:
            return float("nan")
        mu_pos, mu_neg = pos.mean(0), neg.mean(0)
        mu_all = embs.mean(0)
        sb = 0.5 * (np.sum((mu_pos - mu_all)**2) + np.sum((mu_neg - mu_all)**2))
        sw = 0.5 * (pos.var(0).sum() + neg.var(0).sum())
        return float(sb / (sw + 1e-8))

    top5 = df_ok.nlargest(5, "f1")
    eq_rows = []
    for _, row in top5.iterrows():
        feat_p = Path(row["feat_onnx"]) if row.get("feat_onnx") else None
        if feat_p is None or not feat_p.exists():
            print(f"  {row['featurizer']}/{row['loss']}/{row['augment']}: feat ONNX missing")
            continue
        try:
            embs = _extract_embs(feat_p, _wavs_eq)
            fisher = _fisher_ratio(embs, _labels_arr)
            sil = silhouette_score(embs, _labels_arr) if len(np.unique(_labels_arr)) > 1 else float("nan")
            eq_rows.append({
                "featurizer": row["featurizer"],
                "loss": row["loss"],
                "augment": row["augment"],
                "f1": row["f1"],
                "fisher_ratio": round(fisher, 4),
                "silhouette": round(sil, 4),
            })
            print(f"  {row['featurizer']:12s} {row['loss']:8s} {row['augment']:10s}  "
                  f"F1={row['f1']:.3f}  Fisher={fisher:.3f}  Sil={sil:.3f}")
        except Exception as e:
            print(f"  Error for {row['featurizer']}/{row['loss']}: {e}")

    if eq_rows:
        df_eq = pd.DataFrame(eq_rows)
        print()
        print("Embedding quality summary:")
        print(df_eq[["featurizer","loss","augment","f1","fisher_ratio","silhouette"]].to_string(index=False))

        # Bar chart: F1 vs Fisher vs Silhouette for top-5
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        labels_plot = [
            f"{r['featurizer']}\n{r['loss']}\n{r['augment']}" for r in eq_rows
        ]
        for ax, metric, color, title in [
            (axes[0], "f1", "steelblue", "F1"),
            (axes[1], "fisher_ratio", "coral", "Fisher ratio (higher=better)"),
            (axes[2], "silhouette", "seagreen", "Silhouette score (higher=better)"),
        ]:
            vals = [r[metric] for r in eq_rows]
            ax.bar(range(len(eq_rows)), vals, color=color, edgecolor="white")
            ax.set_xticks(range(len(eq_rows)))
            ax.set_xticklabels(labels_plot, fontsize=7)
            ax.set_title(title)
            ax.set_ylabel(metric)

        plt.suptitle(f"Embedding quality — top-5 configs — {WAKE_WORD!r}", fontsize=11)
        plt.tight_layout()
        eq_path = str(Path(OUTPUT_DIR) / "embedding_quality.png")
        plt.savefig(eq_path, dpi=120, bbox_inches="tight")
        plt.show()
        print(f"Embedding quality chart saved: {eq_path}")

## Cell 9 — Best configuration & final verification

Picks the single highest-F1 configuration from the whole grid, prints its full metrics, and
verifies that its two ONNX files (featurizer + classifier head) exist. It then runs one
end-to-end inference on a real positive sample using `OnnxWakeWordInferencer` (the same
runtime path OVOS uses) as a final smoke test — `score > 0.5` prints `PASS`.

Finally it prints a ready-to-copy CLI command and lists the saved plots
(`ablation_heatmap.png`, `augmentation_impact.png`, `embedding_quality.png`).

### Where to go next

- **Train the winner properly.** The ablation runs are short (`EPOCHS=20`) to keep the grid
  affordable. Once you have chosen featurizer + loss + augmentation, retrain that single
  configuration with more epochs and more positives (see the quickstart) for a
  deployment-grade model.
- **Deploy it.** Drop the featurizer and head ONNX into the OVOS
  `ovos-ww-plugin-precise-onnx` plugin (the quickstart's "Ship it" section has the config
  snippet).
- **Go deeper on featurizers.** `nb08_wakehubert.ipynb` swaps in a distilled-HuBERT
  featurizer; `nb10_audioset_featurizer.ipynb` trains a brand-new featurizer on AudioSet.

In [ ]:
import json
import csv
import numpy as np
import torchaudio
from pathlib import Path
from ww_trainer.inference import OnnxWakeWordInferencer

# ── Best config summary + ONNX verification ───────────────────────────────────

if df_ok.empty:
    print("No successful cells yet.")
else:
    best = df_ok.nlargest(1, "f1").iloc[0]

    print("=" * 60)
    print(f"Best ablation configuration — {WAKE_WORD!r}")
    print(f"  Featurizer : {best['featurizer']}")
    print(f"  Loss       : {best['loss']}")
    print(f"  Augment    : {best['augment']}")
    print(f"  F1         : {best['f1']:.4f}")
    print(f"  Precision  : {best['precision']:.4f}")
    print(f"  Recall     : {best['recall']:.4f}")
    print()

    # ONNX verification
    feat_p = Path(best["feat_onnx"]) if best.get("feat_onnx") else None
    head_p = Path(best["head_onnx"]) if best.get("head_onnx") else None

    for label, p in [("featurizer", feat_p), ("head", head_p)]:
        if p and p.exists():
            print(f"  OK  {label}: {p.name}  ({p.stat().st_size/1024:.0f} KB)")
        else:
            print(f"  MISSING  {label}: {p}")

    # Inference test
    if feat_p and feat_p.exists() and head_p and head_p.exists():
        inferencer = OnnxWakeWordInferencer(str(feat_p), str(head_p))
        _pos_path = None
        with open(test_csv) as f:
            for row in csv.reader(f):
                if len(row) >= 2 and row[1].strip() == "1" and Path(row[0]).exists():
                    _pos_path = row[0]; break
        if _pos_path:
            wav, sr = torchaudio.load(_pos_path)
            if sr != 16000:
                wav = torchaudio.functional.resample(wav, sr, 16000)
            score = inferencer.infer(wav.mean(0).numpy().astype(np.float32))
            print(f"\nInference test: score={score:.4f}  ({'PASS' if score > 0.5 else 'LOW'})")

    print()
    print("CLI commands:")
    if feat_p and feat_p.exists() and head_p and head_p.exists():
        print(f"  .venv/bin/python scripts/eval/test_wakeword.py \\")
        print(f"      --featurizer {feat_p} \\")
        print(f"      --model      {head_p} \\")
        print(f"      --audio      sample.wav")
    print()
    print("Output files:")
    for f in [
        Path(OUTPUT_DIR) / "ablation_heatmap.png",
        Path(OUTPUT_DIR) / "augmentation_impact.png",
        Path(OUTPUT_DIR) / "embedding_quality.png",
    ]:
        if f.exists():
            print(f"  {f}")
    print("=" * 60)